In [ ]:
# Ikigai SDK
from ikigai import Ikigai
# Standard library
import warnings

# Third-party
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import urllib3
import yaml
from utils import EXPLAINABILITY_METRICS_CONTEXT, app_info, upload_dataset, build_or_update_flow, run_flow



In [ ]:
# ── Credentials ──────────────────────────────────────────────────────────
with open("key_prod.yaml", "r") as file:
    keys = yaml.safe_load(file)

API_KEY = keys["API_KEY"]
BASE_URL = keys["BASE_URL"]
USER_EMAIL = keys["USER_EMAIL"]

headers = {"User": USER_EMAIL, "Api-key": API_KEY}
ikigai = Ikigai(USER_EMAIL, API_KEY, BASE_URL)




In [ ]:
app_name = "Celonis PoC"
try:
    app = ikigai.apps[app_name]
except RuntimeError:
    app = ikigai.app.new(name=app_name).build()


# Upload dataset

In [ ]:
df = pd.read_csv("celonis_data_forecast.csv")
df.head()

In [ ]:
dataset = upload_dataset(app, "celonis_raw", df)
# Access the dataset on Ikigai platform
app.datasets["celonis_raw"].df().head()

# Preprocessing Flows

## 1. Data cleaning flow

In [ ]:
# Build the data cleaning flow: IMPORTED -> PYTHON (select columns) -> EXPORTED

flow_builder = ikigai.builder
facet_types = ikigai.facet_types

# 1. Import facet — read celonis_raw
dataset = app.datasets["celonis_raw"]
import_facet = flow_builder.facet(
    facet_type=facet_types.INPUT.IMPORTED, name="celonis_raw"
).arguments(
    dataset_id=dataset.dataset_id,
    file_type="csv",
    header=True,
    use_raw_file=False,
)

# 2. Python facet — select only the columns we need
python_script = """
import pandas as pd

columns_used = ['Id', 'NextReleaseDate', 'OldHorizonIssueDate', 'HorizonIssueDate',
       'ReleaseDate', 'DateTimeCreated', 'InternalPartNumber',
       'KeytoTRLP', 'ABCCode', 'Release', 'ReleaseQuantityNet', 'ReleaseQuantityCum',
       'OldRelease', 'OldQuantity', 'ShippedQ']

result = data[columns_used]
# remove rows with NaNs in these important columns
no_nans_columns = ["InternalPartNumber", "ReleaseQuantityNet", "ReleaseDate",]
result = result.dropna(subset=no_nans_columns)
result["ReleaseDate"] = pd.to_datetime(result["ReleaseDate"])
result["ABCCode"] = result["ABCCode"].fillna("Unknown")
# remove duplicates
result = result.drop_duplicates()

# filter out inactive parts (no releases after 2025-01-01)
last_active = result.groupby("InternalPartNumber").agg({"ReleaseDate": "max"}).reset_index()
active_dates_threshold = pd.to_datetime("2025-01-01")
active_parts = last_active[last_active["ReleaseDate"] >= active_dates_threshold]["InternalPartNumber"]
result = result[result["InternalPartNumber"].isin(active_parts)]
result = result.loc[result["ReleaseDate"] >= "2023-05-31"]

"""

python_facet = flow_builder.facet(
    facet_type=facet_types.MID.PYTHON, name="Clean Data"
).arguments(
    script=python_script,
).add_arrow(import_facet, name="data")

# 3. Export facet — save the cleaned dataset
export_name = "celonis_cleaned"
flow_builder.facet(
    facet_type=facet_types.OUTPUT.EXPORTED, name=export_name
).arguments(
    dataset_name=export_name,
    file_type="csv",
    header=True,
    directory="",
).add_arrow(python_facet, name="result")

# Build and run the flow
cleaning_flow = flow_builder.build()

result = run_flow(app, "cleaning_flow", cleaning_flow, ["celonis_cleaned"], high_volume=False)


## 2. Data aggregation flow

In [ ]:
# Build the aggregation flow: IMPORTED -> PYTHON (aggregate) -> 2x EXPORTED

flow_builder = ikigai.builder
facet_types = ikigai.facet_types

# 1. Import facet — read celonis_cleaned
dataset = app.datasets["celonis_cleaned"]
import_facet = flow_builder.facet(
    facet_type=facet_types.INPUT.IMPORTED, name="celonis_cleaned"
).arguments(
    dataset_id=dataset.dataset_id,
    file_type="csv",
    header=True,
    use_raw_file=False,
)

# 2. Python facet — deduplicate and aggregate by week
python_script = """
import pandas as pd

# get only the last record for each part number and release date, based on HorizonIssueDate
df_last = data.sort_values("HorizonIssueDate").groupby(["ReleaseDate", "InternalPartNumber"]).last().reset_index()
# generate W-Mon date range
df_last["week_start"] = pd.to_datetime(df_last["ReleaseDate"]).dt.to_period("W-MON").apply(lambda r: r.start_time)
# aggregate by week and internal part number and sum ReleaseQuantityNet
df_agg = df_last.groupby(["week_start", "InternalPartNumber", "ABCCode"]).agg({"ReleaseQuantityNet": "sum"}).reset_index()
# filter out the last week since it may be incomplete
last_week_start = df_last["week_start"].max()
df_agg = df_agg[df_agg["week_start"] < last_week_start]
result = {"last": df_last, "weekly": df_agg}
"""

python_facet = flow_builder.facet(
    facet_type=facet_types.MID.PYTHON, name="Aggregate Data"
).arguments(
    script=python_script,
).add_arrow(import_facet, name="data")

# 3. Export facet — save the last-observation dataset
export_last_name = "celonis_last"
flow_builder.facet(
    facet_type=facet_types.OUTPUT.EXPORTED, name=export_last_name
).arguments(
    dataset_name=export_last_name,
    file_type="csv",
    header=True,
    directory="",
).add_arrow(python_facet, name="last")

# 4. Export facet — save the weekly aggregated dataset
export_weekly_name = "celonis_weekly"
flow_builder.facet(
    facet_type=facet_types.OUTPUT.EXPORTED, name=export_weekly_name
).arguments(
    dataset_name=export_weekly_name,
    file_type="csv",
    header=True,
    directory="",
).add_arrow(python_facet, name="weekly")

# Build and run the flow
aggregation_flow = flow_builder.build()

result = run_flow(app, "aggregation_flow", aggregation_flow, ["celonis_last", "celonis_weekly"], high_volume=False)

## 3. Aux data flow

In [ ]:
# Build the aux data flow: IMPORTED -> PYTHON (lead-time signals + melt) -> EXPORTED

flow_builder = ikigai.builder
facet_types = ikigai.facet_types

# 1. Import facet — read celonis_cleaned
dataset = app.datasets["celonis_cleaned"]
import_facet = flow_builder.facet(
    facet_type=facet_types.INPUT.IMPORTED, name="celonis_cleaned"
).arguments(
    dataset_id=dataset.dataset_id,
    file_type="csv",
    header=True,
    use_raw_file=False,
)

# 2. Python facet — generate lead-time signals and melt
python_script = """
import pandas as pd
df = data.copy()
df["ReleaseDate"] = pd.to_datetime(df["ReleaseDate"])
df["week_start"] = df["ReleaseDate"].dt.to_period("W-MON").apply(lambda r: r.start_time)
df["LeadTimeDays"] = (pd.to_datetime(df["ReleaseDate"]) - pd.to_datetime(df["HorizonIssueDate"])).dt.days

# Build base with actual (no lead filter)
df_last_all = df.sort_values("HorizonIssueDate").groupby(["ReleaseDate", "InternalPartNumber", "week_start"]).last().reset_index()
df_base = df_last_all.groupby(["week_start", "InternalPartNumber", "ABCCode"]).agg({"ReleaseQuantityNet": "sum"}).reset_index()
df_base = df_base.rename(columns={"ReleaseQuantityNet": "ReleaseQuantityNet_0"})
# reindex to ensure all weeks/parts are present, fill missing with 0
all_weeks = pd.date_range(start=df_base["week_start"].min(), end=df_base["week_start"].max()+pd.DateOffset(weeks=10), freq="W-TUE")
# 
all_parts = df_base["InternalPartNumber"].unique()
all_abc = df_base["ABCCode"].unique()
full_index = pd.MultiIndex.from_product([all_weeks, all_parts, all_abc], names=["week_start", "InternalPartNumber", "ABCCode"])
df_base = df_base.set_index(["week_start", "InternalPartNumber", "ABCCode"]).reindex(full_index, fill_value=0).reset_index()
for lead_time_threshold in [7, 14, 21, 28, 35, 42, 45]:
    df_threshold = df[df["LeadTimeDays"] >= lead_time_threshold]
    df_last = df_threshold.sort_values("HorizonIssueDate").groupby(["ReleaseDate", "InternalPartNumber", "week_start"]).last().reset_index()
    df_agg = df_last.groupby(["week_start", "InternalPartNumber", "ABCCode"]).agg({"ReleaseQuantityNet": "sum"}).reset_index()
    df_agg = df_agg.rename(columns={"ReleaseQuantityNet": f"ReleaseQuantityNet_{lead_time_threshold}"})
    df_base = df_base.merge(df_agg, on=["week_start", "InternalPartNumber", "ABCCode"], how="left")

df_base.fillna(0, inplace=True)

# Melt into long format: date, value, identifier
lead_cols = [c for c in df_base.columns if c.startswith("ReleaseQuantityNet_")]
df_melted = df_base.melt(
    id_vars=["week_start", "InternalPartNumber", "ABCCode"],
    value_vars=lead_cols,
    var_name="identifier",
    value_name="value",
)
df_melted = df_melted.rename(columns={"week_start": "date"})

# Clean up identifier names
df_melted["identifier"] = df_melted["identifier"].str.replace("ReleaseQuantityNet_", "lead_", regex=False)
df_melted["value"] = df_melted["value"].astype(float)

result = df_melted
"""

python_facet = flow_builder.facet(
    facet_type=facet_types.MID.PYTHON, name="Generate Lead Signals"
).arguments(
    script=python_script,
).add_arrow(import_facet, name="data")

# 3. Export facet — save the melted aux dataset
export_name = "aux"
flow_builder.facet(
    facet_type=facet_types.OUTPUT.EXPORTED, name=export_name
).arguments(
    dataset_name=export_name,
    file_type="csv",
    header=True,
    directory="",
).add_arrow(python_facet, name="result")

# Build and run the flow
aux_flow = flow_builder.build()

result = run_flow(app, "aux_flow", aux_flow, ["aux"], high_volume=False)

In [ ]:
pd.infer_freq(app.datasets["celonis_weekly"].df()["week_start"].unique())

### Multi-horizon AiCast forecasts (1w, 2w, 3w, 5w, 8w)

In [ ]:
# ── Multi-horizon forecast builder ────────────────────────────────────────
# Builds AiCast forecast flows for horizons: 1, 2, 3, 5, 8 weeks.
# Each flow: IMPORTED(weekly) → FILTERBYTIME ──────────────────┐
#            IMPORTED(aux)   → Filter(lead ≥ h*7 days) ──┐    │
#                                                    AI_CAST → EXPORTED
# For 8w there are no aux leads ≥ 56 days, so no aux is attached.

from utils.model_util import ensure_model

flow_builder = ikigai.builder
facet_types = ikigai.facet_types
model_types = ikigai.model_types

HORIZONS = [1, 2, 3, 4, 5, 8]
MODEL_TYPE = model_types.AI_CAST.base
PRIMARY_DATASET = "celonis_weekly"
AUX_DATASET = "aux"
DATE_FROM = "2023-05-31"
DATE_TO = "2025-12-31"

# Available lead-time thresholds (days) in the aux dataset
AVAILABLE_LEADS = [0, 7, 14, 21, 28, 35, 42, 45]


def _aux_identifiers_for_horizon(horizon_weeks):
    """Return list of aux lead identifiers whose lag >= horizon in days.
    E.g. 3 weeks → 21 days → lead_21, lead_28, lead_35, lead_42, lead_45
    Returns empty list if none qualify (e.g. 8 weeks = 56 days > max lead).
    """
    min_days = horizon_weeks * 7
    return [f"lead_{d}" for d in AVAILABLE_LEADS if d >= min_days and d < min_days + 7]


def build_forecast_flow(app, horizon):
    """Build a single AiCast forecast flow for the given horizon (weeks)."""
    fb = ikigai.builder

    model_name = f"celonis_aicast_{horizon}w"
    export_name = f"celonis_forecast_{horizon}w"
    flow_name = f"forecast_{horizon}w_flow"

    # Recreate model
    try:
        app.models[model_name].delete()
    except Exception:
        pass
    ensure_model(app, model_name, MODEL_TYPE)

    # 1. Import — primary data
    primary_ds = app.datasets[PRIMARY_DATASET]
    primary_import = fb.facet(
        facet_type=facet_types.INPUT.IMPORTED, name="primary"
    ).arguments(
        dataset_id=primary_ds.dataset_id,
        file_type="csv",
        header=True,
        use_raw_file=False,
    )

    # 2. Filter by time
    time_filter = fb.facet(
        facet_type=facet_types.MID.FILTERBYTIME, name="Filter Primary"
    ).arguments(
        column_name="week_start",
        from_date=DATE_FROM,
        to_date=DATE_TO,
    ).add_arrow(primary_import)

    # 3. (Optional) Aux data — import + filter to eligible leads
    aux_leads = _aux_identifiers_for_horizon(horizon)
    aux_filter = None
    if aux_leads:
        aux_ds = app.datasets[AUX_DATASET]
        aux_import = fb.facet(
            facet_type=facet_types.INPUT.IMPORTED, name="aux"
        ).arguments(
            dataset_id=aux_ds.dataset_id,
            file_type="csv",
            header=True,
            use_raw_file=False,
        )
        # Build filter expression: ([identifier]=='lead_21')|([identifier]=='lead_28')|...
        filter_expr = "|".join(f"([identifier]=='{lid}')" for lid in aux_leads)
        aux_filter = fb.facet(
            facet_type=facet_types.MID.Filter, name=f"Filter Aux ≥{horizon*7}d"
        ).arguments(
            expressions=[filter_expr],
        ).add_arrow(aux_import)

    # 4. AiCast
    model_facet = fb.model_facet(
        facet_type=facet_types.MID.AI_CAST,
        model_type=MODEL_TYPE,
        name=model_name,
    ).add_arrow(time_filter, input_type="primary")
    models_to_include = [ "Sma", "Croston", "Lgmt1", "Tsfm0",
                "Lasso", "Lgbm","Random_Forest","Last_Interval","Lgm-S"
            ]
    if aux_filter is not None:
        model_facet = model_facet.add_arrow(aux_filter, input_type="auxiliary")
        models_to_include += ["Smax", "Holt_Wintersx"]
    model_facet = (
        model_facet
        .arguments(model_name=model_name)
        .hyperparameters(
            type="base",
            hierarchical_type="bottom_up",
            fill_missing_values="zero",
            drop_threshold=1,
            include_reals=True,
            nonnegative=True,
            return_all_levels=False,
            enable_artifacts_saving=False,
            holdout_period=max(2*horizon, 12),
            interval_to_predict=horizon,
            metric="weighted_mean_absolute_percentage_error",
            recalculate_metrics=False,
            best_model_only=True,
            computation_budget=100,
            confidence=0.7,
            enable_conformal_interval=False,
            enable_parallel_processing=False,
            eval_method="cv",
            models_to_include=models_to_include,
            time_budget=10_000,
            enable_guardrails=True,
            guardrail_type="outlier_correction",
            pruner="statistical",
        )
        .parameters(
            identifier_columns=["InternalPartNumber"],
            time_column="week_start",
            value_column="ReleaseQuantityNet",
            mode="train",
        )
    )

    # 5. Export
    model_facet.facet(
        facet_type=facet_types.OUTPUT.EXPORTED, name=export_name
    ).arguments(
        dataset_name=export_name,
        file_type="csv",
        header=True,
        directory="",
    )

    return flow_name, fb.build(), export_name


# ── Build all flow definitions ────────────────────────────────────────────
flow_specs = {}  # {horizon: (flow_name, flow_def, export_name)}
for h in HORIZONS:
    aux_leads = _aux_identifiers_for_horizon(h)
    flow_name, flow_def, export_name = build_forecast_flow(app, h)
    flow_specs[h] = (flow_name, flow_def, export_name)
    aux_info = f"aux: {', '.join(aux_leads)}" if aux_leads else "no aux"
    print(f"✓ Built {h}w forecast flow → {export_name}  ({aux_info})")

print(f"\n{len(flow_specs)} flows ready. Run the next cell to execute them in parallel.")

### Run all forecast flows in parallel

In [ ]:
# ── Run all forecast flows in parallel ─────────────────────────────────────
from concurrent.futures import ThreadPoolExecutor, as_completed
from utils.flow_util import build_or_update_flow
import time

def _run_one(app, flow_name, flow_def, export_name, high_volume=True):
    """Submit, run, and return result for a single forecast flow."""
    while True:
        try:
            flow = build_or_update_flow(app, flow_name, flow_def)
            flow.update_high_volume_preference(high_volume)
            print(f"created flow {flow_name}")
            break
        except:
            continue
    run_log = flow.run()
    if run_log.status != "SUCCESS":
        raise RuntimeError(f"{flow_name} failed: {run_log.data}")
    return flow_name, export_name

t0 = time.time()
results = {}
errors = {}

with ThreadPoolExecutor(max_workers=3) as pool:
    futures = {
        pool.submit(_run_one, app, fn, fd, en): h
        for h, (fn, fd, en) in flow_specs.items()
    }
    for future in as_completed(futures):
        h = futures[future]
        try:
            flow_name, export_name = future.result()
            results[h] = export_name
            elapsed = time.time() - t0
            print(f"✓ {h}w done ({elapsed:.0f}s) → {export_name}")
        except Exception as exc:
            errors[h] = str(exc)
            print(f"✗ {h}w FAILED: {exc}")

elapsed = time.time() - t0
print(f"\n{'─'*50}")
print(f"Completed: {len(results)}/{len(flow_specs)} flows in {elapsed:.0f}s")
if errors:
    print(f"Errors: {errors}")

In [ ]:
# ── Consolidation flow: merge all forecast horizons ───────────────────────
# Imports all 5 forecast datasets, picks shortest-horizon model per (part, week).

flow_builder = ikigai.builder
facet_types = ikigai.facet_types

# 1. Import facets — one per horizon
import_facets = {}
for h in HORIZONS:
    ds_name = f"celonis_forecast_{h}w"
    ds = app.datasets[ds_name]
    arrow_name = f"forecast_{h}w"
    import_facets[h] = flow_builder.facet(
        facet_type=facet_types.INPUT.IMPORTED, name=arrow_name
    ).arguments(
        dataset_id=ds.dataset_id,
        file_type="csv",
        header=True,
        use_raw_file=False,
    )

# 2. Python facet — consolidate: pick shortest-horizon model for each week
#    Build the forecast_sources list dynamically from HORIZONS
sources_literal = ", ".join(f'({h}, data["forecast_{h}w"])' for h in sorted(HORIZONS))

consolidate_script = f"""
import pandas as pd

forecast_sources = [{sources_literal}]

dfs = []
for horizon, f in sorted(forecast_sources, key=lambda x: x[0]):
    f = f.copy()
    f["week_start"] = pd.to_datetime(f["week_start"])
    cutoff = f.loc[f["prediction_type"] == "real", "week_start"].max()
    pred = f.loc[
        (f["prediction_type"] == "predicted_value") & (f["week_start"] > cutoff)
    ].copy()
    pred["week_offset"] = ((pred["week_start"] - cutoff).dt.days / 7).round().astype(int)
    pred["horizon"] = horizon
    dfs.append(pred)

all_preds = pd.concat(dfs, ignore_index=True)
all_preds = all_preds[all_preds["week_offset"] <= all_preds["horizon"]]

# Pick shortest horizon (most specialised) for each (identifier, week)
result = (
    all_preds
    .sort_values("horizon")
    .groupby(["InternalPartNumber", "week_start"])
    .first()
    .reset_index()
)
"""

consolidate_facet = flow_builder.facet(
    facet_type=facet_types.MID.PYTHON, name="Consolidate Forecasts"
).arguments(script=consolidate_script)

# Wire all import arrows
for h in HORIZONS:
    arrow_name = f"forecast_{h}w"
    consolidate_facet = consolidate_facet.add_arrow(import_facets[h], name=arrow_name)

# 3. Export
export_name = "celonis_forecast_consolidated"
flow_builder.facet(
    facet_type=facet_types.OUTPUT.EXPORTED, name=export_name
).arguments(
    dataset_name=export_name,
    file_type="csv",
    header=True,
    directory="",
).add_arrow(consolidate_facet, name="result")

# Build and run
consolidation_flow = flow_builder.build()
result = run_flow(app, "consolidation_flow", consolidation_flow, [export_name], high_volume=False)

In [ ]:
sources_literal

In [ ]:
# Build the evaluation flow:
# IMPORTED (forecast) + IMPORTED (raw) -> PYTHON (join & compute WMAPE) -> EXPORTED

flow_builder = ikigai.builder
facet_types = ikigai.facet_types

# 1. Import facet — consolidated forecast results
forecast_ds = app.datasets["celonis_forecast_consolidated"]
forecast_import = flow_builder.facet(
    facet_type=facet_types.INPUT.IMPORTED, name="celonis_forecast"
).arguments(
    dataset_id=forecast_ds.dataset_id,
    file_type="csv",
    header=True,
    use_raw_file=False,
)

# 2. Import facet — raw data with baselines
raw_ds = app.datasets["celonis_raw"]
raw_import = flow_builder.facet(
    facet_type=facet_types.INPUT.IMPORTED, name="celonis_raw"
).arguments(
    dataset_id=raw_ds.dataset_id,
    file_type="csv",
    header=True,
    use_raw_file=False,
)

# 3. Python facet — join forecasts, compute WMAPE
eval_script = """
import pandas as pd

forecast_df = data["forecast_data"]
df_baselines = data["raw_data"].copy()

# Aggregate raw baselines by week
df_last = df_baselines.sort_values("HorizonIssueDate").groupby(["ReleaseDate", "InternalPartNumber"]).last().reset_index()
df_last["week_start"] = pd.to_datetime(df_last["ReleaseDate"]).dt.to_period("W-MON").apply(lambda r: r.start_time)
df_last["ABCCode"] = df_last["ABCCode"].fillna("Unknown")
df_agg = df_last.groupby(["week_start", "InternalPartNumber", "ABCCode"]).agg({
    "ReleaseQuantityNet": "sum",
    "LightGBMForecast": "mean",
    "ProphetForecast": "mean"
}).reset_index()

# Filter to test period (7 weeks)
test_date = "2025-12-31"
test_end = pd.to_datetime(test_date) + pd.DateOffset(weeks=7)
df_agg_test = df_agg.loc[(df_agg["week_start"] > test_date) & (df_agg["week_start"] <= test_end)]

# Get aicast forecast predictions (consolidated — already filtered to predicted_value)
test_forecasts = forecast_df.copy()
test_forecasts["week_start"] = pd.to_datetime(test_forecasts["week_start"])
# If consolidated data still has prediction_type column, filter to predicted_value
if "prediction_type" in test_forecasts.columns:
    test_forecasts = test_forecasts.loc[test_forecasts["prediction_type"] == "predicted_value"].copy()
# Limit to 7 weeks
test_forecasts = test_forecasts.loc[test_forecasts["week_start"] <= test_end]
df_agg_test["week_start"] = pd.to_datetime(df_agg_test["week_start"])

# Join all forecasts
all_forecasts = test_forecasts[["InternalPartNumber", "week_start", "ReleaseQuantityNet"]].copy()
# Carry week_offset from consolidated forecast if available
if "week_offset" in test_forecasts.columns:
    all_forecasts["week_offset"] = test_forecasts["week_offset"].values
all_forecasts.rename(columns={"ReleaseQuantityNet": "aicast_forecast"}, inplace=True)
all_forecasts = all_forecasts.set_index(["InternalPartNumber", "week_start"])
forecast_values = df_agg_test.set_index(["InternalPartNumber", "week_start"]).to_dict()
all_forecasts["ProphetForecast"] = all_forecasts.index.map(forecast_values["ProphetForecast"])
all_forecasts["LightGBMForecast"] = all_forecasts.index.map(forecast_values["LightGBMForecast"])
all_forecasts["actual"] = all_forecasts.index.map(forecast_values["ReleaseQuantityNet"])
all_forecasts_nona = all_forecasts.dropna()

# Compute WMAPE
def wmape(actual, predicted):
    return (actual - predicted).abs().sum() / max(actual.abs().sum(), 1)

results = {}
for col in ["aicast_forecast", "ProphetForecast", "LightGBMForecast"]:
    results[col] = wmape(all_forecasts_nona["actual"], all_forecasts_nona[col])

wmape_df = pd.DataFrame.from_dict(results, orient="index", columns=["WMAPE"])
wmape_df = wmape_df.reset_index().rename(columns={"index": "model"})

# Compute per-identifier WMAPE and add as columns
all_out = all_forecasts_nona.reset_index()
for col in ["aicast_forecast", "ProphetForecast", "LightGBMForecast"]:
    per_part = all_out.groupby("InternalPartNumber").apply(
        lambda g: (g["actual"] - g[col]).abs().sum() / max(g["actual"].abs().sum(), 1)
    ).rename(f"wmape_{col}")
    all_out = all_out.merge(per_part, on="InternalPartNumber", how="left")

# Compute WMAPE per week_offset
wmape_per_offset_rows = []
if "week_offset" in all_out.columns:
    for offset in sorted(all_out["week_offset"].unique()):
        subset = all_out[all_out["week_offset"] == offset]
        row = {"week_offset": int(offset)}
        for col in ["aicast_forecast", "ProphetForecast", "LightGBMForecast"]:
            row[f"wmape_{col}"] = wmape(subset["actual"], subset[col])
        wmape_per_offset_rows.append(row)
wmape_per_offset = pd.DataFrame(wmape_per_offset_rows)

result = {"wmape": wmape_df, "all_forecasts": all_out, "wmape_per_offset": wmape_per_offset}
"""

eval_facet = flow_builder.facet(
    facet_type=facet_types.MID.PYTHON, name="Evaluate Forecasts"
).arguments(
    script=eval_script,
).add_arrow(forecast_import, name="forecast_data").add_arrow(raw_import, name="raw_data")

# 4. Export facet — WMAPE results
export_wmape = "celonis_wmape"
flow_builder.facet(
    facet_type=facet_types.OUTPUT.EXPORTED, name=export_wmape
).arguments(
    dataset_name=export_wmape,
    file_type="csv",
    header=True,
    directory="",
).add_arrow(eval_facet, name="wmape")

# 5. Export facet — all forecasts joined
export_all = "celonis_all_forecasts"
flow_builder.facet(
    facet_type=facet_types.OUTPUT.EXPORTED, name=export_all
).arguments(
    dataset_name=export_all,
    file_type="csv",
    header=True,
    directory="",
).add_arrow(eval_facet, name="all_forecasts")

# 6. Export facet — WMAPE per week offset
export_offset = "celonis_wmape_per_offset"
flow_builder.facet(
    facet_type=facet_types.OUTPUT.EXPORTED, name=export_offset
).arguments(
    dataset_name=export_offset,
    file_type="csv",
    header=True,
    directory="",
).add_arrow(eval_facet, name="wmape_per_offset")

# Build and run the flow
eval_flow = flow_builder.build()

result = run_flow(app, "eval_flow", eval_flow, ["celonis_wmape", "celonis_all_forecasts", "celonis_wmape_per_offset"], high_volume=False)

In [ ]:
app.datasets["celonis_wmape_per_offset"].df()

In [ ]:
app.datasets["celonis_wmape"].df()